## Import and Define Values

In [1]:
import pandas as pd
import glob
import os

all_files = sorted(glob.glob(r'E:\Data_wow\pipeline_test\data_sample\*.parquet'))
df = pd.read_parquet(all_files[0])

## Explore Sample Data

In [10]:
print(f"File: {os.path.basename(all_files[0])}")
print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nHead:\n{df.head()}")

File: 2023-01-01_00-00-00.parquet
Shape: (1914, 5)

Dtypes:
department_name            object
sensor_serial              object
create_at          datetime64[ns]
product_name               object
product_expire     datetime64[ns]
dtype: object

Head:
                    department_name  \
0  qvgtdfuivcdsboxnraqpokjzaayedfuy   
1  qvgtdfuivcdsboxnraqpokjzaayedfuy   
2  qvgtdfuivcdsboxnraqpokjzaayedfuy   
3  qvgtdfuivcdsboxnraqpokjzaayedfuy   
4  qvgtdfuivcdsboxnraqpokjzaayedfuy   

                                       sensor_serial  create_at  \
0  ogaacjdszqnmihmbmowurxertkvisyxlvcckouzohjgjmn... 2023-01-01   
1  jlcupyajaxwljjbtpgibuqkwldckyzitvtuxycxrasefba... 2023-01-01   
2  ktzwxsgymyxdlsxtkbipvokudhlgbmdjgayiemhpzvctfb... 2023-01-01   
3  lfrmstkemkybgehgybdaxwqvcbdsczowunbpgvlsgpzrye... 2023-01-01   
4  ggmpdkynngneislqxzjviszeskllebikdfnfxeaskknlli... 2023-01-01   

       product_name product_expire  
0  ukdvwjumzjdffmmk     2023-03-31  
1  wtmvatzvplvekyyu     2023-03-29  


## Null check and Basic check

In [ ]:
print("nulls:")
print(df.isnull().sum())

print()
print(df.describe(include='all', datetime_is_numeric=True))

nulls:
department_name    0
sensor_serial      0
create_at          0
product_name       0
product_expire     0
dtype: int64

                         department_name  \
count                               1914   
unique                               100   
top     xyzbvwdydfyvohgkaqbubahuvnycitid   
freq                                  29   

                                            sensor_serial  \
count                                                1914   
unique                                               1876   
top     nuvirigkmbhzrgjwlvpvrsnthcgnyufxjtoajonmbsqhvy...   
freq                                                    3   

             product_name       product_expire  
count                1914                 1914  
unique                856                    3  
top     offpcjsovaqdswki  2023-03-29 00:00:00  
freq                   8                  660  


## Unique col

In [13]:
print("unique counts:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique values")

print(f"\nEach file has {len(df)} rows (1 row per sensor per minute)")
print(f"department_name length: {df['department_name'].str.len().unique()} chars")
print(f"sensor_serial length:   {df['sensor_serial'].str.len().unique()} chars")
print(f"product_name length:    {df['product_name'].str.len().unique()} chars")

unique counts:
  department_name: 100 unique values
  sensor_serial: 1876 unique values
  create_at: 1 unique values
  product_name: 856 unique values
  product_expire: 3 unique values

Each file has 1914 rows (1 row per sensor per minute)
department_name length: [32] chars
sensor_serial length:   [64] chars
product_name length:    [16] chars


In [12]:
df['product_expire'].value_counts()

2023-03-29    660
2023-03-30    655
2023-03-31    599
Name: product_expire, dtype: int64

## Department and Sensor relationship

In [2]:
sensors_per_dept = df.groupby('department_name')['sensor_serial'].nunique()
print("sensors per dept:")
print(sensors_per_dept.describe())
print(f"\nmin: {sensors_per_dept.min()}, max: {sensors_per_dept.max()}, total depts: {sensors_per_dept.shape[0]}, total sensors: {sensors_per_dept.sum()}")

# just verifying sensor <-> dept is consistent across the data
sensor_dept_map = df.groupby('sensor_serial')['department_name'].nunique()
print(f"\nsensors that appear in >1 dept: {(sensor_dept_map > 1).sum()}")

sensors per dept:
count    100.000000
mean      18.760000
std        6.502711
min        5.000000
25%       14.000000
50%       19.000000
75%       25.000000
max       29.000000
Name: sensor_serial, dtype: float64

min: 5, max: 29, total depts: 100, total sensors: 1876

sensors that appear in >1 dept: 0


## Data Scale

In [17]:
total_files = len(all_files)
rows_per_file = len(df)
estimated_total_rows = total_files * rows_per_file

print(f"total files:  {total_files:,}")
print(f"rows/file:    {rows_per_file:,}")
print(f"total rows:  {estimated_total_rows:,}")

first_file = os.path.basename(all_files[0]).replace('.parquet', '')
last_file  = os.path.basename(all_files[-1]).replace('.parquet', '')
print(f"\nFirst file: {first_file}")
print(f"Last file:  {last_file}")

print(f"\nproduct_expire range: {df['product_expire'].min()} to {df['product_expire'].max()}")

total files:  43,201
rows/file:    1,914
total rows:  82,686,714

First file: 2023-01-01_00-00-00
Last file:  2023-01-31_00-00-00

product_expire range: 2023-03-29 00:00:00 to 2023-03-31 00:00:00


## SUMMARY

Going with a single flat table  all 5 columns load as-is into one table. (Faster)

- table: `sensor_readings` — ~82M rows
- columns: `department_name`, `sensor_serial`, `create_at`, `product_name`, `product_expire`
